# Small Deep Model — 1D-CNN over byte-pair tokens

Owner: Jana

Bake-off candidate B (research.md Decision 1). Trains a compact 1D-CNN sized to keep inference under the 50 ms p95 budget after ONNX export. Saved as a temporary `.pt` artifact in this directory; `export_onnx.py` converts it to `../artifacts/classifier.onnx`. Runs on Colab GPU; `torch` is allowed in training but banned from the serving image (Principle I).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import f1_score

CLASSES = ["SPAM", "FAQ", "ACCOUNT_OPS", "HARD_QUESTION", "UNKNOWN"]
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}

train = pd.read_csv("data/cleaned/clean_strict_train.csv")
val = pd.read_csv("data/cleaned/clean_strict_val.csv")

# Char-level hashed features keep the input dimensionality fixed for ONNX export.
vec = HashingVectorizer(analyzer="char_wb", ngram_range=(2, 4), n_features=2**14, alternate_sign=False, norm="l2")
X_train = torch.from_numpy(vec.transform(train["message"]).toarray()).float()
X_val = torch.from_numpy(vec.transform(val["message"]).toarray()).float()
y_train = torch.tensor([CLASS_IDX[c] for c in train["label"]])
y_val = torch.tensor([CLASS_IDX[c] for c in val["label"]])


class TinyCNN(nn.Module):
    def __init__(self, in_dim: int, n_classes: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyCNN(X_train.shape[1], len(CLASSES)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    opt.zero_grad()
    logits = model(X_train.to(device))
    loss = loss_fn(logits, y_train.to(device))
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(X_val.to(device)).argmax(dim=1).cpu().numpy()
    print(epoch, "val macro-F1:", f1_score(y_val, pred, average="macro"))

torch.save({"state_dict": model.state_dict(), "in_dim": X_train.shape[1]}, "classifier_dl.pt")
print("Wrote classifier_dl.pt")
